In [1]:
from google.colab import drive
import torch
import os
import glob
import numpy as np
import cv2

In [2]:
#Mount Google Drive and navigate to the assignment directory
drive.mount('/content/drive')
%cd /content/drive/MyDrive/assignments/Video_action_recognition
!git pull
%pip install -q -r requirements.txt
%env PYTHONPATH=src
%env UCF50_N_FRAMES=16
with open("/content/drive/MyDrive/credentials/wandb_api.txt") as f:
    os.environ["WANDB_API_KEY"] = f.read().strip()
print("W&B key loaded:", len(os.environ.get("WANDB_API_KEY", "")) > 0)
# Check for PyTorch and CUDA availability
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Mounted at /content/drive
/content/drive/MyDrive/assignments/Video_action_recognition
Already up to date.
env: PYTHONPATH=src
env: UCF50_N_FRAMES=16
W&B key loaded: True
PyTorch version: 2.11.0+cu130
CUDA available: True


In [ ]:
# Setup paths for dataset
drive_path = '/content/drive/MyDrive/datasets/ucf50'
local_path = '/content'
url = 'https://www.crcv.ucf.edu/data/UCF50/UCF50.rar'
extract_path = os.path.join(local_path, 'datasets')
raw_dir = os.path.join(extract_path, 'UCF50')
frame_dir = os.path.join(extract_path,'UCF50_frames')

In [4]:
!python scripts/prep_dataset.py --drive_path {drive_path} --local_path {local_path} --url {url}

Copying rar file from Google Drive to local path.
Copied rar file from Google Drive to local path.
Extracting rar file to /content/datasets/UCF50...
Extracted rar file to /content/datasets/UCF50.


In [5]:
video_counts = {}
frame_counts = []
failed_videos = []
for category in os.listdir('/content/datasets/UCF50'):
    category_path = os.path.join('/content/datasets/UCF50', category)
    video_count = 0
    for video in os.listdir(category_path):
        video_path = os.path.join(category_path, video)
        v_cap = cv2.VideoCapture(video_path)
        if not v_cap.isOpened():
            print("Failed to open video:", video_path)
            failed_videos.append(video_path)
            v_cap.release()
            continue
        v_len = int(v_cap.get(cv2.CAP_PROP_FRAME_COUNT))
        v_cap.release()
        frame_counts.append(v_len)
        video_count += 1
    video_counts[category] = video_count
largest_category = max(video_counts, key=video_counts.get)
smallest_category = min(video_counts, key=video_counts.get)
print("Number of videos:", sum(video_counts.values()))
print("Largest category:", largest_category, video_counts[largest_category])
print("Smallest category:", smallest_category, video_counts[smallest_category])
print("Median frame count:", np.median(frame_counts))
print("Min frame count:", np.min(frame_counts))
print("Max frame count:", np.max(frame_counts))
print("Failed videos:", failed_videos)

Number of videos: 6681
Largest category: HorseRiding 197
Smallest category: TaiChi 100
Median frame count: 181.0
Min frame count: 29
Max frame count: 900
Failed videos: []


In [6]:
n_frames = 16
!python scripts/extract_frames.py --raw_dir {raw_dir} --n_frames {n_frames} --frame_dir {frame_dir}

Converting from avi: 100% 6681/6681 [14:16<00:00,  7.80file/s]
Extracted 6681 and skipped 0 out of 6681 avis


In [7]:
!python -m pytest -v -rs tests/

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/drive/MyDrive/assignments/Video_action_recognition
configfile: pytest.ini
plugins: typeguard-4.6.0, anyio-4.15.1, langsmith-0.14.11, platformdirs-4.13.3
collected 10 items                                                             

tests/test_dataset.py::test_raw_class_folders PASSED                     [ 10%]
tests/test_dataset.py::test_video_avis PASSED                            [ 20%]
tests/test_dataset.py::test_ucf50_frames_dir PASSED                      [ 30%]
tests/test_dataset.py::test_video_dirs PASSED                            [ 40%]
tests/test_dataset.py::test_frame_counts PASSED                          [ 50%]
tests/test_dataset.py::test_frames_read PASSED                           [ 60%]
tests/test_models.py::test_batch_independence PASSED                     [ 70%]
te

In [8]:
!bash scripts/train.sh --dropout 0.3 --n_epochs 50 --weight_decay 1e-1 --freeze_layers conv1 bn1 layer1 layer2 layer3 --num_workers 7 --run_name jitter_do03_aug_wd1e-1_freeze3

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth
100% 83.3M/83.3M [00:00<00:00, 209MB/s]
Loading video dataset....
100% 50/50 [00:00<00:00, 5884.26it/s]
Splitting train/validation/test datasets....
Dropping 63 videos in groups outside 1-25
Train groups: [ 1  2  3  4  7  9 10 12 13 14 15 16 17 18 19 21 22 23 24]
Validation groups: [ 6 20]
Test groups: [ 5  8 11 25]
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: ttl074 (ttl074-johns-hopkins-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run why1h5yq (0.3s)
wandb: ⣯ setting up run why1h5yq (0.3s)
wandb: ⣟ setting up run why1h5yq (0.3s)
wandb: ⡿ setting up run why1h5yq (0.3s)
wandb: ⢿ set

: 